# Retrieving Data with Python

## Introduction

In the Python Basics notebooks, whenever we needed data, we defined variables manually. For example, let's look at the following function. It prints a simple P&L. It requires the P&L to be given as a dictionary, in USD billions.

In [1]:
def print_pnl(pnl_dict, company=None, year=None):
    if company is None:
        company = pnl_dict['company']
    if year is None:
        year = pnl_dict['year']
    print(f"P&L Summary for {company}, year {year} (USD billions)")
    print(f"Total Revenue:     {pnl_dict['total_revenue']:>10.2f}")
    print(f"Cost of Revenue:   {pnl_dict['cost_of_revenue']:>10.2f}")
    print(f"Gross Profit:      {pnl_dict['gross_profit']:>10.2f}")
    print(f"EBITDA:            {pnl_dict['ebitda']:>10.2f}")

If we want to use it, we can fill the dictionary manually as a variable, and call the function:

In [2]:
dict_amzn_2024_pnl = {
    "company": "AMZN",
    "year": 2024,
    "total_revenue": 637.96,
    "cost_of_revenue": 326.29,
    "gross_profit": 311.67,
    "ebitda": 123.82
}
print_pnl(dict_amzn_2024_pnl)

P&L Summary for AMZN, year 2024 (USD billions)
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82


The main issues with this are that: a) you need to know some Python to change the variable and run the code again, and b) it's only doable for small amounts of data. While this works for simple exercises, real-world programs almost never rely on manually typed data. Instead, they retrieve data from external sources—files, databases, APIs, or even user input.

In this notebook, we explore several common ways to bring external data into Python. We begin with the simplest type of input: data provided directly by the user. Then move on to reading files, working with structured formats like JSON, accessing spreadsheets, scraping websites, and calling APIs.

## Data inputs

In this first case we look at how to get data directly from the user. The data is not located anywhere we can read, and needs to provided by the person running the script. We could ask that person to modify a variable to store the data, but that would require them knowing some Python basics. This approach is used more often in scripts than notebooks, but we can still use it here.

In [3]:
year = input("Enter the year to generate the report: ")
print("Generating report for year: ", year)

Enter the year to generate the report:  2023


Generating report for year:  2023


We can use input several times, and integrate it as part of loops and other conditions. For example, if we want to generate a report for an undetermined number of years, and the user has to provide them:

In [4]:
dict_pnl = {
    "company": None,
    "year": None,
    "total_revenue": None,
    "cost_of_revenue": None,
    "gross_profit": None,
    "ebitda": None
}

for key in dict_pnl.keys():
    value = input(f"Enter value for {key}: ")
    if key == "company":
        dict_pnl[key] = value
    elif key == "year":
        dict_pnl[key] = int(value)
    else:
        dict_pnl[key] = float(value)
        
print_pnl(dict_pnl)

Enter value for company:  AMZN
Enter value for year:  2024
Enter value for total_revenue:  637.96
Enter value for cost_of_revenue:  326.29
Enter value for gross_profit:  311.67
Enter value for ebitda:  123.82


P&L Summary for AMZN, year 2024 (USD billions)
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82


Remember we are learning tools at our disposal. We could even use a mix of variables and inputs to store the different years, and ask the user which year to print:

In [5]:
dict_amzn_2023_pnl = {
    "company": "AMZN",
    "year": 2023,
    "total_revenue": 574.79,
    "cost_of_revenue": 304.74,
    "gross_profit": 270.05,
    "ebitda": 89.4
}
year = input("Select P&L year for AMZN:")
if year == "2024":
    print_pnl(dict_amzn_2024_pnl)
if year == "2023":
    print_pnl(dict_amzn_2023_pnl)
else:
    print("P&L not available.")

Select P&L year for AMZN: 2024


P&L Summary for AMZN, year 2024 (USD billions)
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82
P&L not available.


## Reading files with Python

One of the simplest ways to retrieve large amounts of data is using files. If it is too long to copy to a variable, store it in a file. If you want the user to change the content without interacting with Python, store it in file. Another advantage of obtaining data from files is that other sources might generate the files that you want to extract information from. For example, several programs and webs have an "Export" function that stores data in files. If we open files with a specific format (like an Excel file) we will need to use specific code for them. Let's look first at how we can read text files. 

Note: files like JSON or CSV are text files, but since we organize the data in a predetermined way, there are better ways to open them that we will explore later.

In [6]:
file_name = "amzn_pnl.txt"

If you prefer it, you can use a local file. The following cell contains the code to retrieve the file from the GitHub repository:

In [7]:
import requests
def import_github_example_file(file):
    url = f'https://raw.githubusercontent.com/D-G-D/Python-Bootcamp/refs/heads/main/02%20Data%20Manipulation/{file_name}'
           
    response = requests.get(url)
    response.raise_for_status()  # Raises an error if the download fails
    with open(file, "wb") as f:
        f.write(response.content)
    print(f"Downloaded '{file}' successfully.")

In [8]:
import_github_example_file(file_name)

Downloaded 'amzn_pnl.txt' successfully.


Now that you have the file in your local folder, let's look how to open it.

In [9]:
f = open(file_name, "r")

`f` is a file object. `"r"` means read-only, so we can modify the file. We can read from it in several ways. For example, reading the entire file as a single string:

In [10]:
entire_file = f.read()
print(entire_file)

P&L Summary for AMZN (USD billions)

Year 2024
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82

Year 2023
Total Revenue:         574.79
Cost of Revenue:       304.74
Gross Profit:          270.05
EBITDA:                 89.40


Now, before we see another way of reading the file, we need to understand a key thing about this file object: it has a cursor pointing to the next line to read. Thus, when we read the file after having read it all, there is nothing else. Let's test it:

In [11]:
entire_file = f.read()
print(entire_file)

There are two ways to start over. One, open the file again. In that case, it's better to close the file before you open it again to prevent issues with your operating system having multiple open accesses to the same file. Two, using `seek` to move to the initial position:

In [12]:
f.seek(0)

0

In [13]:
entire_file = f.read()
print(entire_file)
f.close()

P&L Summary for AMZN (USD billions)

Year 2024
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82

Year 2023
Total Revenue:         574.79
Cost of Revenue:       304.74
Gross Profit:          270.05
EBITDA:                 89.40


In [14]:
f = open(file_name, "r")
entire_file = f.read()
print(entire_file)

P&L Summary for AMZN (USD billions)

Year 2024
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82

Year 2023
Total Revenue:         574.79
Cost of Revenue:       304.74
Gross Profit:          270.05
EBITDA:                 89.40


Reading line by line, and adding line numbers:

In [15]:
f = open(file_name, "r")
for line_number, line in enumerate(f, start=1):
    print(f"{line_number}: {line.strip()}")
f.close()

1: P&L Summary for AMZN (USD billions)
2: 
3: Year 2024
4: Total Revenue:         637.96
5: Cost of Revenue:       326.29
6: Gross Profit:          311.67
7: EBITDA:                123.82
8: 
9: Year 2023
10: Total Revenue:         574.79
11: Cost of Revenue:       304.74
12: Gross Profit:          270.05
13: EBITDA:                 89.40


Most Python users will use `with`, a context manager, to open a file. This ensures that the file is closed at the end of executing the code, without having to manually care of it, and even if there are erros:

In [16]:
with open(file_name, "r") as f:
    for line_number, line in enumerate(f, start=1):
        print(f"{line_number}: {line.strip()}")

1: P&L Summary for AMZN (USD billions)
2: 
3: Year 2024
4: Total Revenue:         637.96
5: Cost of Revenue:       326.29
6: Gross Profit:          311.67
7: EBITDA:                123.82
8: 
9: Year 2023
10: Total Revenue:         574.79
11: Cost of Revenue:       304.74
12: Gross Profit:          270.05
13: EBITDA:                 89.40


You might have noticed that when we retrieved the GitHub file using `requests`, we used `with open` with the file name, but `"wb"` instead of `"r"`. This wrote the retrieved content into the local file. This is called the mode, and it can be a combination of letters and symbols:
- `b`: Binary.
- `t`: Text (default, so barely used)
- `r`: Opens the file for reading. File must exist.
- `w`: Opens the file for writing. Truncates file (replaces all the content).
- `a`: Opens the file for writing, only appending at the end. The cursor starts at the end of the file.
- `x`: Creates the file, fails if it already exists. Permission to write the file.
- `+`: Combined with the other letters, it adds both permissions to read and write.

## JSON Files

JSON (JavaScript Object Notation) is a commonly used text-based format used to store and exchange structured data that doesn't follow a strict schema. It is structured because data is organized using key–value pairs and nested collections such as lists and objects. Unlike a table found in SQL databases or Excel files, the content varies from record to record, allowing different entries to have different fields and levels of nesting. Let's see an example:

In [17]:
file_name = "amzn_pnl.json"
import_github_example_file(file_name)
with open(file_name, "r") as f:
    print(f.read())

Downloaded 'amzn_pnl.json' successfully.
{
	"company": "AMZN",
	"pnl": [
		{
			"year": 2023,
			"total_revenue": 574.79,
			"cost_of_revenue": 304.74,
			"gross_profit": 270.05,
			"ebitda": 89.40
		},
		{
			"year": 2024,
			"total_revenue": 637.96,
			"cost_of_revenue": 326.29,
    			"gross_profit": 311.67,
			"ebitda": 123.82
		}
	]
}



As you can see, there is a hierarchy in the file, with each year P&L ressembling the dict objects we defined earlier. We were able to read the file using the syntax we learnt in the previous section, but that is not the best option that we have. If we use `open`, we treat everything as text and store the file we have read as a string, without acknowledging the structure of a JSON file. It is more efficient to use the `json` module that will return a nested structure composed of dictionaries, lists, strings, numbers, booleans and nulls:

In [18]:
import json
with open(file_name, "r") as f:
    json_pnl = json.load(f)
print(json_pnl)
print(type(json_pnl))

{'company': 'AMZN', 'pnl': [{'year': 2023, 'total_revenue': 574.79, 'cost_of_revenue': 304.74, 'gross_profit': 270.05, 'ebitda': 89.4}, {'year': 2024, 'total_revenue': 637.96, 'cost_of_revenue': 326.29, 'gross_profit': 311.67, 'ebitda': 123.82}]}
<class 'dict'>


This dictionary contains the P&Ls of years 2023 and 2024. If we want to search a specific year...

In [19]:
def json_pnl_print_year(target_year):
    company = json_pnl["company"]
    for one_year in json_pnl["pnl"]:
        # We are selecting each of the elements in the list "pnl", that currently consists of two dictionaries.
        if one_year["year"] == target_year:
            print_pnl(one_year, company=company)

In [20]:
json_pnl_print_year(2023)

P&L Summary for AMZN, year 2023 (USD billions)
Total Revenue:         574.79
Cost of Revenue:       304.74
Gross Profit:          270.05
EBITDA:                 89.40


In [21]:
json_pnl_print_year(2024)

P&L Summary for AMZN, year 2024 (USD billions)
Total Revenue:         637.96
Cost of Revenue:       326.29
Gross Profit:          311.67
EBITDA:                123.82


If instead of a file, we want to read a string (or a variable containing a string), we use `json.loads()`. APIs often return data in JSON format, as a string, and not a file. Note that JSON strings require the quotes to be double quotes. For example:

In [22]:
string_2023 = '{"company":"AMZN","year":2023,"total_revenue":574.79,"cost_of_revenue":326.29,"gross_profit":270.05, \
               "operating_income":68.59,"pretax_income":68.61,"net_income":59.25}'
json_amzn_2023_pnl = json.loads(string_2023)
print(json_amzn_2023_pnl["year"])

2023


## Reading CSV with Pandas

CSV format is one of the most common formats for exporting/importing table-like data between systems. Being text-based, even an editor like Notepad can open a CSV file. The acronym CSV stands for "Comma-Separated Values", because each column is separated by a comma. For example:
<code><br>name, age
david, 60
</code> 

If the content of a column contains a string with a comma, quotes are used to differentiate between a comma character, and the column separator.

In [23]:
file_name = "amzn_pnl.csv"
import_github_example_file(file_name)
with open(file_name, "r") as f:
    print(f.read())

Downloaded 'amzn_pnl.csv' successfully.
company,year,total_revenue,cost_of_revenue,gross_profit,ebitda
AMZN,2023,574.79,304.74,270.05,89.40
AMZN,2024,637.96,326.29,311.67,123.82



Since the comma is used in many countries as a decimal separator, some CSV files use semicolons `;` as column separators. Another character that is often used is the *tab*. You will sometimes see files ending with the extension TSV (Tab-Separated Values) but people also use the CSV extension for tab-separated data.

Given the tabular nature of the data, CSV files in Python are often manipulated using **Pandas**. We will go deeper into the analysis in different sessions, focusing here only on loading the data. The pandas object is called a **DataFrame** and is often stored in a variable named `df`:

In [24]:
import pandas as pd
df = pd.read_csv(file_name)
print(df)

  company  year  total_revenue  cost_of_revenue  gross_profit  ebitda
0    AMZN  2023         574.79           304.74        270.05   89.40
1    AMZN  2024         637.96           326.29        311.67  123.82


Pandas auto-generates the index with a sequence starting with 0. We can also specify a column:

In [25]:
pd.read_csv(file_name, index_col="year")

,company,total_revenue,cost_of_revenue,gross_profit,ebitda
year,,,,,
2023,AMZN,574.79,304.74,270.05,89.40
2024,AMZN,637.96,326.29,311.67,123.82


Pandas is assuming that the first row are the column names. If not, we can specify it:

In [26]:
pd.read_csv(file_name, header=None)

,0,1,2,3,4,5
0,company,year,total_revenue,cost_of_revenue,gross_profit,ebitda
1,AMZN,2023,574.79,304.74,270.05,89.40
2,AMZN,2024,637.96,326.29,311.67,123.82


We can also see the column data types, selected by pandas based on the content. We can also force the type.

In [27]:
df = pd.read_csv(file_name)
df.dtypes

company             object
year                 int64
total_revenue      float64
cost_of_revenue    float64
gross_profit       float64
ebitda             float64
dtype: object

Finally, some common options for `read_csv`:
- Using a different separator, for example tab: `sep="\t"`
- File using special encoding: `encoding="latin1"`
- Avoiding dates loading as strings: `parse_dates=["date_column_name"]`
- Wrong data type guesses: `dtype={"column_name": str}`

## Excel files

Excel files are binary. If you open them with a text editor like notepad, or `open` in text mode... they are not human-readable. Unlike CSV files, Excel files contain multiple sheets, formulas, metadata and cell formatting. If we are only interested in the data contained in Excel, we can use pandas to extract the values. For example, the file `amzn.xlsx` in the course GitHub has two sheets, one per year. If we want to retrieve the data:

In [28]:
file_name = "amzn.xlsx"
import_github_example_file(file_name)

Downloaded 'amzn.xlsx' successfully.


In [29]:
df = pd.read_excel(file_name)
print(df)

   total_revenue  cost_of_revenue  gross_profit  ebitda
0         574.79           304.74        270.05    89.4


Now, this obtained the first sheet. What if we want a specific sheet? And... what sheets does the file have?

In [30]:
xlsx = pd.ExcelFile(file_name)
xlsx.sheet_names
xlsx.close()

In [31]:
df = pd.read_excel(file_name, sheet_name="2024")
print(df)

   total_revenue  cost_of_revenue  gross_profit  ebitda
0         637.96           326.29        311.67  123.82


What if we want to obtain any information about the formulas or formatting? Then instead of using Pandas, we use `openpyxl`

In [32]:
from openpyxl import load_workbook

wb = load_workbook(file_name, data_only=False) # workbook
ws = wb.active # worksheet
gross_profit_header = ws["C1"]
gross_profit_val = ws["C2"]

In [33]:
print(gross_profit_header.value)
print(gross_profit_val.value)
print(gross_profit_header.font.bold)
print(gross_profit_val.font.bold)

gross_profit
=A2-B2
True
False


If we want to retrieve the *last saved* values, we need to open with `data_only=True`. Note that `openpyxl` doesn't calculate the results of the formulas, it only takes the values stored in the file.

## Web Scraping

Web scraping is the process of retrieving data from websites automatically, by downloading the code that generates the web. The key about web scraping is that it reproduces what a person would do, much faster. This means we could, for example, scrape the latest market status, the latest news, or the weather status almost instantaneously, many times a day. The problem with this is that too many *bots* (web scraping machines) could collapse the web server. Thus, websites have terms of service establishing what they allow, and take action to prevent users sending too many requests. Thus, if you decide to do some scraping, be responsible.

To download a web page, we use the module `requests`. We have seen this already when retrieving the files from the course's GitHub. We were, in fact, scraping.

In [34]:
import requests

url = "https://example.com"
response = requests.get(url)
response.raise_for_status()

html_code = response.text
print(html_code)

<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.<p><a href="https://iana.org/domains/example">Learn more</a></div></body></html>



The code above is HTML. It uses tags to tell your browser how the text between the tags should be displayed. For example, the tag `<a>` is used for hyperlinks. To make sense of all this we parse it using `BeautifulSoup`. For example, let's find the links:

In [35]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_code, "html.parser")

In [36]:
links = soup.find_all("a")
for link in links:
    print(link.get("href"))

https://iana.org/domains/example


Now, let's count the number of containers:

In [37]:
containers = soup.find_all("div")
print(len(containers))

1


A problem with this method is that many pages today are dynamically generated with JavaScript and thus the final version shown to the user doesn't match the code you obtain with request. Most pages using JavaScript will have a backend API from where they obtain information; you might be able to obtain access to it. Another alternative is to use tools that can execute JavaScript to scrape them after being rendered, like `selenium`. Again, make sure that anything you do is compliant with the pages' terms and conditions.

In some cases, enough people want to scrape a certain page that specific Python code is created. Let's see an example of using `yfinance` to get data from Yahoo Finance, specifically, AMZN market data for 2022. Not all Python distributions come with `yfinance`, so we will install it first:

In [38]:
%%capture
!pip install yfinance

In [39]:
import yfinance as yf
import datetime

ticker_name = "AMZN"
start = datetime.datetime.utcfromtimestamp(1640995200)
end = datetime.datetime.utcfromtimestamp(1672444800)
df = yf.download(
    ticker_name,
    start=start,
    end=end,
    interval="1d",
    auto_adjust=False
)

if isinstance(df.columns, pd.MultiIndex): # If MultiIndex, flatten it.
    df.columns = [col[0] for col in df.columns]

[*********************100%***********************]  1 of 1 completed


In [40]:
df

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2022-01-03,170.404495,170.404495,170.703506,166.160507,167.550003,63520000
2022-01-04,167.522003,167.522003,171.399994,166.349503,170.438004,70726000
2022-01-05,164.356995,164.356995,167.126495,164.356995,166.882996,64302000
2022-01-06,163.253998,163.253998,164.800003,161.936996,163.450500,51958000
2022-01-07,162.554001,162.554001,165.243500,162.031006,163.839005,46606000
...,...,...,...,...,...,...
2022-12-23,85.250000,85.250000,85.779999,82.930000,83.250000,57433700
2022-12-27,83.040001,83.040001,85.349998,83.000000,84.970001,57284000
2022-12-28,81.820000,81.820000,83.480003,81.690002,82.800003,58228600


## About APIs

An alternative to Web Scraping are APIs. An API (Application Programming Interface) is a structured way for programs to communicate with each other. They usually provide the information in JSON format. Thus, there is no need to extract the data from a human readable form (rendered HTML) to a clean data structure. The drawback is that they often have a monetary cost.

APIs should always be used when available. They are endpoints designed to obtain information, often using an authentication key, providing access to information without overloading the website. Most API providers have guides and support provided to developers wanting to use their APIs. That should be your starting point.

## 